In [1]:
print("all ok")

all ok


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [7]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3:latest",
    temperature=0
)

response = llm.invoke("Explain Kubernetes in simple words in 2 lines")
print(response.content)

Here's a simple explanation of Kubernetes:

Kubernetes is a way to manage and orchestrate multiple containers (like Docker) that run your application, so you don't have to worry about how they're running or scaling. It helps keep your containers healthy, replicates them if one fails, and makes it easy to update or move them around as needed.


In [ ]:
from typing_extensions import TypedDict, Annotated
import operator

from langchain_core.messages import AnyMessage, HumanMessage, AIMessage

In [ ]:
class GraphState(TypedDict):
    message: Annotated[list[AnyMessage], operator.add]

In [ ]:
def llm_call(state: GraphState) -> dict:
    """Call the LLM using conversation messages and append AI response."""
    response = llm.invoke(state["messages"])  # AIMessage
    return {
        "messages": [response]
    }

In [ ]:
def token_counter(state: GraphState) -> dict:
    """Count tokens (simple word count) in the last AI message."""
    last_msg = state["messages"][-1]
    text = last_msg.content
    token_number = len(text.split())
    summary = f"Total token number in the generated answer (word count) is {token_number}"
    return {
        "messages": [AIMessage(content=summary)]
    }

In [ ]:
from langgraph import StateGraph

builder = StateGraph(GraphState)
builder.add_node("llm_call", llm_call)
builder.add_node("token_counter", token_counter)
builder.set_entrypoint("llm_call")
builder.add_edge("llm_call", "token_counter")
builder.set_finish_point("token_counter")